In [1]:
import cmdstanpy as csp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

c:\Users\sebas\one\OneDrive\Namizje\repos\BayesianStats\hw3\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from cmdstanpy import install_cmdstan

install_cmdstan(dir="C:/cmdstan", overwrite=True, compiler=True)

12:08:23 - cmdstanpy - INFO - Add C++ toolchain to $PATH: C:\cmdstan\RTools40


CmdStan install directory: C:/cmdstan
Installing CmdStan version: 2.37.0
Download successful, file: C:\Users\sebas\AppData\Local\Temp\tmpt20wqf03
Extracting distribution
Unpacked download as cmdstan-2.37.0
Building version cmdstan-2.37.0, may take several minutes, depending on your system.
Overwrite requested, remove existing build of version cmdstan-2.37.0
Rebuilding version cmdstan-2.37.0
Installed cmdstan-2.37.0
Test model compilation


True

In [5]:
model = csp.CmdStanModel(stan_file="..\models\\moving_average.stan", force_compile=True)

12:39:57 - cmdstanpy - INFO - compiling stan file C:\Users\sebas\one\OneDrive\Namizje\repos\BayesianStats\hw3\models\moving_average.stan to exe file C:\Users\sebas\one\OneDrive\Namizje\repos\BayesianStats\hw3\models\moving_average.exe
12:40:15 - cmdstanpy - INFO - compiled model executable: C:\Users\sebas\one\OneDrive\Namizje\repos\BayesianStats\hw3\models\moving_average.exe


In [6]:
df = pd.read_csv("../no2.csv")
df["day"] = df.index + 1

In [7]:
seasonality = 7

In [8]:
stan_data = {
    "n": len(df),
    "t": df["day"],
    "y": df["no2"],
    "k": 2,
    "omega": np.array([2*np.pi/seasonality, 2*np.pi/365.25])
}


In [ ]:
fit = model.sample(
    data=stan_data,
    chains=4,
    parallel_chains=4,
    iter_warmup=1000,
    iter_sampling=1000,
    seed=42,
)

12:40:16 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]


chain 1:   0%|          | 1/2000 [00:00<22:11,  1.50it/s, (Warmup)]


chain 1:   5%|▌         | 100/2000 [00:46<14:41,  2.16it/s, (Warmup)]


chain 1:  15%|█▌        | 300/2000 [07:06<43:32,  1.54s/it, (Warmup)]


chain 1:  35%|███▌      | 700/2000 [12:33<20:06,  1.08it/s, (Warmup)]

chain 1:  60%|██████    | 1200/2000 [17:32<08:10,  1.63it/s, (Sampling)]

chain 1:  80%|████████  | 1600/2000 [21:02<03:34,  1.86it/s, (Sampling)]


chain 1: 100%|██████████| 2000/2000 [24:23<00:00,  1.97it/s, (Sampling)]


















































chain 2: 100%|██████████| 2000/2000 [1:32:32<00:00,  2.78s/it, (Sampling completed)]

chain 3: 100%|██████████| 2000/2000 [1:32:32<00:00,  2.78s/it, (Sampling completed)]


chain 4: 100%|██████████| 2000/2000 [1:32:32<00:00,  2.78s/it, (Sampling completed)]


14:12:48 - cmdstanpy - INFO - CmdStan done processing.
14:12:48 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'moving_average.stan', line 52, column 4 to column 27)
	Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'moving_average.stan', line 52, column 4 to column 27)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'moving_average.stan', line 52, column 4 to column 27)
	Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'moving_average.stan', line 52, column 4 to column 27)
	Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'moving_average.stan', line 52, column 4 to column 27)
	Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'moving_average.stan', line 52, column 4 to column 27)
	Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'moving_average.stan', line 52, column 

14:13:04 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 999 divergent transitions (99.9%)
	Chain 2 had 999 divergent transitions (99.9%)
	Chain 3 had 999 divergent transitions (99.9%)
	Chain 4 had 999 divergent transitions (99.9%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


: 

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np

# 1. Convert CmdStanPy fit to ArviZ InferenceData
idata = az.from_cmdstanpy(posterior=fit, observed_data={"y": stan_data["y"]})

# 2. Sampling quality inspection
# Traceplots for all parameters
az.plot_trace(idata)
plt.tight_layout()
plt.show()

# Summary (Rhat, ESS, mean, sd, hdi)
summary = az.summary(idata, round_to=2)
print(summary)

# 3. Posterior predictive check
# First generate posterior predictive
ppc = model.generate_quantities(
    data=stan_data,
    mcmc_sample=fit,
    gq_output_dir="gq_out"
)

# Convert to InferenceData with posterior_predictive
idata = az.from_cmdstanpy(
    posterior=fit,
    posterior_predictive=ppc,
    observed_data={"y": stan_data["y"]}
)

# 4. Plot posterior predictive vs data
az.plot_ppc(
    idata,
    data_pairs={"y": "y"},
    num_pp_samples=200,  # adjust for speed
    kind="kde"           # or "hist"
)
plt.title("NO2 Posterior Predictive Check")
plt.show()
